In [2]:
import sys, pathlib

root = pathlib.Path.cwd()
if not (root / "src").is_dir():          # walk up until we find src/
    root = next(p for p in root.parents if (p / "src").is_dir())
sys.path.insert(0, str(root / "src"))    # stage_discharge (src-layout)

# sibling project: Python programming/fetch_data
projects = root.parents[1]               # .../Python programming
fd = projects / "fetch_data"
sys.path.insert(0, str(fd / "src") if (fd / "src" / "fetch_data").is_dir() else str(projects))

from stage_discharge.read_write import read_hobo_log, read_vusitu_log, clean_hobo_log, clean_vusitu_log
from stage_discharge.build_stage import calc_diff_pressure, calc_depth, build_stage
from stage_discharge.hobo_fetch import fetch_licor_data, tidy_licor_data
from stage_discharge.rating_curve import RatingCurve
from stage_discharge.conversions import to_units
from stage_discharge.correction import correct_stage
from fetch_data import texmesonet

import pandas as pd
import matplotlib.pyplot as plt

# Dolan Crossing Stage-Discharge Record 10/2026 update
I am compiling and combining the dolan crossing discharge record from the various sources from 2015-2026. Data from 2015-2023 is in the Hunt 2025 Devils River report. 2023-2026 is from RT100 deployed in the river with the data pulled straight fro the sensor files. 2026- is from the Hobo RX system deployed at the site in 4/2026.

I am going to first correct and compile a full stage record working from the oldest data to the newest. Correction will be made based on manual reading of the staff gauge installed in November 2022.

## 2015-2023 stage record
- Big jump in September 2018 is real, there was a large rain event. 
- For some reason the probe stage to staff gage stage calculation was not done for some values, recalculating based on probe_stage - 0.6 as specified in data and report
- check data types, get rid of duplicates
- some uncertainty in what stage measurements are from where....sensor stage record is derived from a constant offset of stage-0.6 to standardize to staff gauge datum, pre 2022 manual stage readings are of unknown provenance...

In [3]:
import os
import pandas as pd

print(os.getcwd())

# ---------- helper: turn a datetime column into a clean, unique, ns DatetimeIndex ----------
def to_datetime_index(df, col, label):
    df = df.copy()
    df.columns = df.columns.str.strip()                     # guard against 'datetime ' etc.
    dt = pd.to_datetime(df[col], errors='coerce')
    if getattr(dt.dt, 'tz', None) is not None:              # tz-aware -> naive local
        dt = dt.dt.tz_localize(None)
    df[col] = dt

    n_bad = df[col].isna().sum()
    if n_bad:
        print(f'[{label}] dropping {n_bad} rows with unparseable datetimes')
    df = df.dropna(subset=[col]).set_index(col).sort_index()

    n_dup = df.index.duplicated().sum()
    if n_dup:
        print(f'[{label}] dropping {n_dup} duplicate timestamps:')
        print(df[df.index.duplicated(keep=False)])
    df = df[~df.index.duplicated(keep='first')]

    df.index = pd.DatetimeIndex(df.index).as_unit('ns')
    df.index.name = 'datetime'
    assert isinstance(df.index, pd.DatetimeIndex) and df.index.is_unique and df.index.is_monotonic_increasing
    return df


# ---------- continuous logger record ----------
# data from hunt report; the xlsx read is ~1m 13s, so cache a pickle and reuse it
xlsx_path = './data/2015_2024_dev300log.xlsx'
pkl_path = './data/2015_2024_dev300log.pkl'
if os.path.exists(pkl_path) and os.path.getmtime(pkl_path) > os.path.getmtime(xlsx_path):
    rt_2015 = pd.read_pickle(pkl_path)
else:
    rt_2015 = pd.read_excel(xlsx_path)
    rt_2015.to_pickle(pkl_path)

# Est Gage: very questionable stage prior to staff gauge install; derived as RT probe stage - 0.6 (= staff gauge reading)
rt_2015_stage = rt_2015[['date', 'Est Gage', 'Stage_ft']].copy()
rt_2015_stage = to_datetime_index(rt_2015_stage, 'date', 'logger')

# force numeric, coercing stray strings/junk to NaN
for c in ['Est Gage', 'Stage_ft']:
    rt_2015_stage[c] = pd.to_numeric(rt_2015_stage[c], errors='coerce')
print(rt_2015_stage.dtypes)

# fill Est Gage gaps from Stage_ft - 0.6 (i think brian just didn't for some reason?)
before = rt_2015_stage['Est Gage'].isna().sum()
rt_2015_stage['Est Gage'] = rt_2015_stage['Est Gage'].fillna(rt_2015_stage['Stage_ft'] - 0.6)
after = rt_2015_stage['Est Gage'].isna().sum()
print(f'Est Gage NaN before fill: {before}, after fill: {after}')
if after:
    print('Remaining NaN rows (Stage_ft also missing):')
    print(rt_2015_stage[rt_2015_stage['Est Gage'].isna()])


# ---------- manual measurements ----------
dev300_stg_q = pd.read_excel('./data/DEV300_stage_discharge.xlsx')
dev300_stg_q = to_datetime_index(dev300_stg_q, 'datetime', 'manual')
for c in ['stage', 'discharge_cfs']:
    dev300_stg_q[c] = pd.to_numeric(dev300_stg_q[c], errors='coerce')


# ---------- sanity check + correct ----------
print(type(rt_2015_stage.index).__name__, rt_2015_stage.index.dtype, len(rt_2015_stage))
print(type(dev300_stg_q.index).__name__, dev300_stg_q.index.dtype, len(dev300_stg_q))

c:\Users\will\OneDrive\Documents\Python programming\stage_discharge\stage_discharge\notebooks\DEV300_dolan
[logger] dropping 1 rows with unparseable datetimes
[logger] dropping 1 duplicate timestamps:
                     Est Gage  Stage_ft
date                                   
2019-02-05 17:30:00   1.10016   1.70016
2019-02-05 17:30:00   1.10016   1.70016
Est Gage    float64
Stage_ft    float64
dtype: object
Est Gage NaN before fill: 32640, after fill: 291
Remaining NaN rows (Stage_ft also missing):
                     Est Gage  Stage_ft
datetime                               
2016-02-25 17:45:00       NaN       NaN
2016-09-27 11:30:00       NaN       NaN
2016-09-27 11:45:00       NaN       NaN
2017-04-24 16:00:00       NaN       NaN
2017-04-24 17:45:00       NaN       NaN
...                       ...       ...
2023-04-18 14:45:00       NaN       NaN
2023-04-18 15:00:00       NaN       NaN
2023-04-18 15:15:00       NaN       NaN
2023-04-18 15:30:00       NaN       NaN
2023-04-18 1

## Stage discharge record

In [4]:
import plotly.graph_objects as go

fig = go.Figure()

# continuous record (WebGL for speed)
fig.add_trace(go.Scattergl(
    x=rt_2015_stage.index,
    y=rt_2015_stage['Est Gage'],
    mode='lines',
    line=dict(width=1.5, color='#2b6cb0'),
    name='Logger (Est Gage)',
))

# manual measurements: also WebGL and added second, so they draw on top
fig.add_trace(go.Scattergl(
    x=dev300_stg_q.index,
    y=dev300_stg_q['stage'],
    mode='markers',
    marker=dict(color='red', size=10, symbol='circle', line=dict(width=1, color='white')),
    name='Manual measurements',
))

fig.update_layout(
    title='DEV300 Dolan Crossing, 2015–2023 (Hunt, 2025 Report)',
    template='plotly_white',
    xaxis_title=None,
    yaxis_title='Stage (ft)',
    font=dict(size=12),
    margin=dict(l=40, r=90, t=50, b=40),
    updatemenus=[dict(
        type='buttons',
        direction='right',
        x=1.0, y=1.15,
        xanchor='right', yanchor='top',
        showactive=True,
        buttons=[
            dict(label='Linear', method='relayout', args=[{'yaxis.type': 'linear', 'yaxis.range': [0, 10]}]),
            dict(label='Log', method='relayout', args=[{'yaxis.type': 'log', 'yaxis.range': [-1, 1]}]),  # 10^-1 = 0.1, 10^1 = 10
        ],
    )],
)

fig.update_xaxes(showgrid=False)
fig.update_yaxes(range=[0, 10], showgrid=True, gridcolor='rgba(0,0,0,0.08)')

fig.write_html('dev300_stage.html', include_plotlyjs='cdn')

# correct record using measurements

Since staff gauge was installed only in 2022, correcting based on the "manual measurements" from before that is made up and silly and only useful for the stage-discharge relationship... the record prior to that is uncorrected. There was one measurement in 2016 that was flagged by brian as errant.

In [5]:
rt_2015_stage_corrected = correct_stage(rt_2015_stage['Est Gage'], dev300_stg_q['stage'][(dev300_stg_q['for_correction'] == 1) & (dev300_stg_q['flagged'] == 0)], extrapolate='zero', tolerance='1.5h')

In [6]:
checks = rt_2015_stage_corrected.attrs['checks']
unmatched = dev300_stg_q.loc[~dev300_stg_q.index.isin(checks.index), ['stage']].copy()

log_idx = rt_2015_stage.index
pos = log_idx.get_indexer(unmatched.index, method='nearest')
unmatched['nearest_logger'] = log_idx[pos]
unmatched['gap'] = (unmatched['nearest_logger'] - unmatched.index).abs()
unmatched['logger_value'] = rt_2015_stage['Est Gage'].iloc[pos].values
print(unmatched.tail(5))

print('logger ends:', log_idx.max())

                     stage          nearest_logger                      gap  \
datetime                                                                      
2022-06-10 08:31:00   0.78 2022-06-10 08:30:00.000          0 days 00:01:00   
2023-12-07 10:59:00   0.82 2023-12-07 08:30:15.475   0 days 02:28:44.525000   
2023-12-08 08:49:00   0.82 2023-12-07 08:30:15.475   1 days 00:18:44.525000   
2023-12-08 09:13:00   0.82 2023-12-07 08:30:15.475   1 days 00:42:44.525000   
2024-05-25 12:33:00   0.63 2023-12-07 08:30:15.475 170 days 04:02:44.525000   

                     logger_value  
datetime                           
2022-06-10 08:31:00      0.749243  
2023-12-07 10:59:00      0.481080  
2023-12-08 08:49:00      0.481080  
2023-12-08 09:13:00      0.481080  
2024-05-25 12:33:00      0.481080  
logger ends: 2023-12-07 08:30:15.475000


In [7]:
rt_2015_stage_corrected.to_csv('./data/corrected_stage_2015.csv')

In [8]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

corr = rt_2015_stage_corrected
checks = corr.attrs.get('checks')

is_corr    = dev300_stg_q['for_correction'] == 1
is_flagged = dev300_stg_q['flagged'] == 1
corr_pts    = dev300_stg_q[is_corr & ~is_flagged]
derived_pts = dev300_stg_q[~is_corr & ~is_flagged]
flagged_pts = dev300_stg_q[is_flagged]

fig = make_subplots(
    rows=2, cols=1, shared_xaxes=True,
    row_heights=[0.85, 0.15], vertical_spacing=0.03,
)

# --- main panel ---
fig.add_trace(go.Scattergl(
    x=corr.index, y=corr['stage_raw'],
    mode='lines', line=dict(width=1.2, color='#a0aec0'),
    name='Uncorrected',
), row=1, col=1)

fig.add_trace(go.Scattergl(
    x=corr.index, y=corr['stage_corrected'],
    mode='lines', line=dict(width=1.5, color='#2b6cb0'),
    name='Corrected',
), row=1, col=1)

fig.add_trace(go.Scattergl(
    x=derived_pts.index, y=derived_pts['stage'],
    mode='markers',
    marker=dict(symbol='circle-open', color='red', size=9, line=dict(width=1.5)),
    name='Derived (sensor − 0.6)',
), row=1, col=1)

fig.add_trace(go.Scattergl(
    x=flagged_pts.index, y=flagged_pts['stage'],
    mode='markers',
    marker=dict(symbol='x', color='#dd6b20', size=10),
    name='Flagged',
), row=1, col=1)

fig.add_trace(go.Scattergl(
    x=corr_pts.index, y=corr_pts['stage'],
    mode='markers',
    marker=dict(color='red', size=10, line=dict(width=1, color='white')),
    name='Staff gauge (used for correction)',
), row=1, col=1)

# --- error strip ---
fig.add_trace(go.Scattergl(
    x=corr.index, y=corr['error'],
    mode='lines', line=dict(width=1, color='#4a5568'),
    showlegend=False,
), row=2, col=1)

if checks is not None and {'stage_raw', 'stage_measured'} <= set(checks.columns):
    fig.add_trace(go.Scattergl(
        x=checks.index, y=checks['stage_raw'] - checks['stage_measured'],
        mode='markers', marker=dict(color='red', size=5),
        showlegend=False,
    ), row=2, col=1)

fig.add_hline(y=0, line=dict(color='rgba(0,0,0,0.3)', width=1, dash='dot'), row=2, col=1)

fig.update_layout(
    title=dict(
        text=(
            'DEV300 Dolan Crossing — Stage, 2015–2024'
            '<br><sup>Logger record corrected to staff-gauge readings from Nov 2022 on; '
            'earlier record is sensor − 0.6 ft (Hunt, 2025)</sup>'
        ),
        x=0.01, xanchor='left', y=0.98, yanchor='top',
        font=dict(size=18),
    ),
    template='plotly_white',
    font=dict(size=12),
    height=780,
    margin=dict(l=55, r=20, t=110, b=40),
    legend=dict(orientation='h', yanchor='bottom', y=1.01, xanchor='right', x=1),
    hovermode='x unified',
)

fig.update_yaxes(title_text='Stage (ft)', range=[0, 10], row=1, col=1)
fig.update_yaxes(title_text='Error (ft)', title_font=dict(size=10), tickfont=dict(size=9), nticks=3, row=2, col=1)
fig.update_xaxes(showgrid=False)
fig.update_yaxes(showgrid=True, gridcolor='rgba(0,0,0,0.08)')

fig.write_html('dev300_stage_error.html', include_plotlyjs='cdn')

# 2023-present(?) RT100 logs
Marcus shared sensor logs which are found in data/sensor logs, these are uncorrected and really just the raw abs and baro pressure. Need to read, clean, build and correct the record.

VuSitu_Log_2023-12-07_10-00-00_Unnamed_231207_dolancrossing
- 12/2023-10/2024 absolute pressure and temperature
- no associated barometric pressure, must pull from texmesonet and correct for elevation offset

VuSitu_Log_2025-10-15_18-00-00_Dolan_Baro_20251015_DolanBaro.csv
- 10/2024-5/2026 barometric pressure

VuSitu_Log_2025-10-15_18-00-00_dolan_xing_tube_20251015_dolanXing.csv
- 10/2024-5/2026 absolute pressure

In [9]:
os.getcwd()

'c:\\Users\\will\\OneDrive\\Documents\\Python programming\\stage_discharge\\stage_discharge\\notebooks\\DEV300_dolan'

In [ ]:
rt_dev300_2023_header, rt_dev300_2023, _ = read_vusitu_log('./data/SensorLogs_fromMOG/VuSitu_Log_2023-12-07_10-00-00_Unnamed_231207_dolancrossing.csv', pressure_kind= 'abs')
rt_dev300_2023_abs = clean_vusitu_log(rt_dev300_2023)
print(rt_dev300_2023_header)

In [20]:
import numpy as np

rt_dev300_2023_baro = texmesonet.get_station(
    station='del norte',
    start='2023-12-07 10:00',
    end='2024-10-31 12:00',
    variables=['Air Pressure', 'Temperature'],
    freq='15min',
)

baro_elevation_m = 581.5584   # Del Norte station, Google Earth
abs_elevation_m = 404.4696    # DEV300 logger, Google Earth
dz = abs_elevation_m - baro_elevation_m   # negative: logger is ~177 m lower

units = rt_dev300_2023_baro.attrs['units']

# mbar -> Pa (TexMesonet TWDB stations report pressure in millibars)
p_unit = units.get('Air Pressure', '')
assert p_unit.lower() in ('mb', 'mbar', 'millibar', 'millibars', 'hpa'), f'unexpected pressure unit: {p_unit!r}'
p_baro = rt_dev300_2023_baro['Air Pressure'] * 100

# air temperature -> K (for the hypsometric equation)
t_unit = units.get('Temperature', '')
t_air = rt_dev300_2023_baro['Temperature']
if 'f' in t_unit.lower():
    t_air = (t_air - 32) * 5 / 9
t_air = t_air.interpolate(limit=8).fillna(15.0)   # fill short gaps; 15 °C standard if missing
t_layer_k = t_air + 273.15 - 0.0065 * dz / 2       # mean temp of the air column between the two sites

# hypsometric equation: shift baro from station elevation to logger elevation
g, Rd = 9.80665, 287.05
rt_dev300_2023_baro = pd.DataFrame(index=rt_dev300_2023_baro.index)
rt_dev300_2023_baro['baro_pressure_pa'] = p_baro * np.exp(-g * dz / (Rd * t_layer_k))
rt_dev300_2023_baro.attrs['units'] = {'baro_pressure_pa': 'Pa'}

print(f'mean elevation correction: +{(rt_dev300_2023_baro["baro_pressure_pa"] - p_baro).mean():.0f} Pa')

mean elevation correction: +1961 Pa


In [16]:
rt_dev300_2023_baro.attrs

{'station': 'Devils River Del Norte',
 'network': 'TWDB',
 'units': {'baro_pressure_pa': 'Pa'},
 'tz': 'UTC',
 'header': {'name': 'Devils River Del Norte',
  'state': 'TX',
  'county': 'Val Verde',
  'latitude': 29.933202,
  'longitude': -100.94142}}

In [21]:
rt_dev300_2023 = pd.merge_asof(
    rt_dev300_2023_abs.sort_index(), rt_dev300_2023_baro.sort_index(),
    left_index=True, right_index=True,
    direction="nearest",
    tolerance=pd.Timedelta("5min"),      # pair a reading only with a baro sample within 1 s
    suffixes=("_abs", "_baro"),
)
    
stage = build_stage(rt_dev300_2023, abs_col='abs_pressure_pa', baro_col='baro_pressure_pa', temp_col='temperature_c')

In [22]:
stage.describe()

,temperature_c,abs_pressure_pa,baro_pressure_pa,diff_pressure_pa,water_level_m
count,31454.000000,31454.000000,29129.000000,29129.000000,29129.000000
mean,22.747845,100401.439536,96836.018973,3731.257917,0.387127
std,6.867827,22992.090164,526.749517,23846.197585,2.502296
min,-79.999650,10330.248249,95195.952984,-86385.592424,-9.861397
25%,19.318512,97599.384268,96481.235972,877.961708,0.089836
50%,24.513260,98084.247712,96828.501353,1306.831010,0.133688
75%,26.706268,99299.129751,97185.671395,2467.270693,0.251963
max,31.083160,384808.288521,98546.056405,287598.301545,35.008650


In [24]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

d = stage.copy()

# flag obvious junk (adjust thresholds as needed)
bad = (
    (d['temperature_c'] < 0) | (d['temperature_c'] > 40) |
    (d['abs_pressure_pa'] < 90_000) | (d['abs_pressure_pa'] > 150_000) |
    (d['diff_pressure_pa'] < 0)
)
print(f'flagged {bad.sum()} of {len(d)} rows; range {d.index.min()} → {d.index.max()}')
d.loc[bad, ['abs_pressure_pa', 'diff_pressure_pa', 'water_level_m']] = float('nan')
d.loc[(d['temperature_c'] < 0) | (d['temperature_c'] > 40), 'temperature_c'] = float('nan')

fig = make_subplots(
    rows=3, cols=1, shared_xaxes=True,
    row_heights=[0.55, 0.25, 0.20], vertical_spacing=0.04,
)

# water level
fig.add_trace(go.Scattergl(
    x=d.index, y=d['water_level_m'], mode='lines',
    line=dict(width=1.2, color='#2b6cb0'), name='Water level',
), row=1, col=1)

# pressures (kPa for readability)
fig.add_trace(go.Scattergl(
    x=d.index, y=d['abs_pressure_pa'] / 1000, mode='lines',
    line=dict(width=1, color='#2c5282'), name='Absolute (logger)',
), row=2, col=1)
fig.add_trace(go.Scattergl(
    x=d.index, y=d['baro_pressure_pa'] / 1000, mode='lines',
    line=dict(width=1, color='#dd6b20'), name='Baro (Del Norte)',
), row=2, col=1)

# temperature
fig.add_trace(go.Scattergl(
    x=d.index, y=d['temperature_c'], mode='lines',
    line=dict(width=1, color='#c53030'), name='Water temp',
), row=3, col=1)

fig.update_layout(
    title=dict(
        text='DEV300 Dolan Crossing — 2023–24 logger record'
             '<br><sup>Barometrically compensated with TexMesonet Del Norte; junk readings removed</sup>',
        x=0.01, xanchor='left', font=dict(size=18),
    ),
    template='plotly_white',
    font=dict(size=12),
    height=800,
    margin=dict(l=60, r=20, t=100, b=40),
    legend=dict(orientation='h', yanchor='bottom', y=1.01, xanchor='right', x=1),
    hovermode='x unified',
)

fig.update_yaxes(title_text='Water level (m)', row=1, col=1)
fig.update_yaxes(title_text='Pressure (kPa)', row=2, col=1)
fig.update_yaxes(title_text='Temp (°C)', row=3, col=1)
fig.update_xaxes(showgrid=False)
fig.update_yaxes(showgrid=True, gridcolor='rgba(0,0,0,0.08)')

fig.write_html('dev300_2023_logger.html', include_plotlyjs='cdn')

flagged 479 of 31454 rows; range 2023-12-07 15:00:00.058000+00:00 → 2024-10-30 06:15:00.058000+00:00
